In [1]:
from collections import Counter, defaultdict
from math import log
import numpy as np

In [2]:
def entropy(y, base=2):
    # e = -p log p -> sum(e)  => [ H(Y) = - \sum_i p_i \log_{base}(p_i) ]
    count = Counter(y)
    ans = 0
    for freq in count.values():
        prob = freq / len(y)
        ans -= prob * log(prob, base)
    return ans


In [3]:
def conditional_entropy(x, y, base=2):
    freq_y_total = defaultdict(Counter)
    # print(freq_y_total)
    freq_x = Counter()
    for i in range(len(x)):
        freq_y_total[x[i]][y[i]] += 1
        # print("i and fyt", i,x[i], y[i], freq_y_total[x[i]][y[i]])
        freq_x[x[i]] += 1
    ans = 0
    # print(freq_y_total)
    # print(freq_x)
    for xi, freq_y_xi in freq_y_total.items():
        res = 0
        for freq in freq_y_xi.values():
            # print(freq, freq_x[xi])
            prob = freq / freq_x[xi]
            res -= prob * log(prob, base)
        ans += res * (freq_x[xi] / len(x))
    return ans


In [4]:

class DecisionTreeI3WithoutPruning:

    class Node:
        def __init__(self, mark, use_feature=None, children=None):
            if children is None:
                children = {}
            self.mark = mark
            # a set of feature indexes that are still allowed to be used for splitting at this node.
            self.use_feature = use_feature
            self.children = children

        @property
        def is_leaf(self):
            return len(self.children) == 0

    def __init__(self, x, y, labels=None, base=2, epsilon=0):
        if labels is None:
            labels = ["feature{}".format(i + 1) for i in range(len(x[0]))]
        self.labels = labels
        self.base = base
        self.epsilon = epsilon

        # ---------- build the tree ----------
        self.n = len(x[0])
        self.root = self._build(x, y, set(range(self.n)))

    def _build(self, x, y, spare_features_idx):
        """
        :param x: input
        :param y: output
        :param spare_features_idx: index for next
        """
        freq_y = Counter(y)

        if len(freq_y) == 1:
            return self.Node(y[0])

        """
        When there are no remaining features to split on, the algorithm stops and creates a leaf node.
        That leaf node is labeled with the majority class in the current data subset.
        """
        # If there are no features left to split on, stop
        if not spare_features_idx:
            return self.Node(freq_y.most_common(1)[0][0])

        # calculate the information gain choose the bigger
        best_feature_idx, best_gain = -1, 0
        for feature_idx in spare_features_idx:
            gain = self.information_gain(x, y, feature_idx)
            if gain > best_gain:
                best_feature_idx, best_gain = feature_idx, gain

        # epsilon is threshold, stop
        if best_gain <= self.epsilon:
            return self.Node(freq_y.most_common(1)[0][0])

        # 依Ag=ai将D分割为若干非空子集Di，将Di中实例数最大的类作为标记，构建子结点
        node = self.Node(freq_y.most_common(1)[0][0], use_feature=best_feature_idx)
        features = set()
        sub_x = defaultdict(list)
        sub_y = defaultdict(list)
        for i in range(len(x)):
            feature = x[i][best_feature_idx]
            features.add(feature)
            sub_x[feature].append(x[i])
            sub_y[feature].append(y[i])

        # print("ffff->", features)
        for feature in features:
            node.children[feature] = self._build(sub_x[feature], sub_y[feature],
                                                 spare_features_idx - {best_feature_idx})
        return node

    def __repr__(self):

        def dfs(node, depth=0, value=""):
            if node.is_leaf:
                res.append(value + " -> " + node.mark)
            else:
                if depth > 0:
                    res.append(value + " :")
                for val, child in node.children.items():
                    dfs(child, depth + 1, "  " * depth + self.labels[node.use_feature] + " = " + val)

        res = []
        dfs(self.root)
        return "\n".join(res)

    def information_gain(self, x, y, idx):
        """计算信息增益"""
        return entropy(y, base=self.base) - conditional_entropy([x[i][idx] for i in range(len(x))], y, base=self.base)

In [5]:
class DecisionTreeC45WithoutPruning(DecisionTreeI3WithoutPruning):
    def information_gain(self, x, y, idx):
        return super().information_gain(x, y, idx) / entropy([x[i][idx] for i in range(len(x))], base=self.base)

In [6]:
X, Y = [np.array([["青年", "否", "否", "一般"],
                  ["青年", "否", "否", "好"],
                  ["青年", "是", "否", "好"],
                  ["青年", "是", "是", "一般"],
                  ["青年", "否", "否", "一般"],
                  ["中年", "否", "否", "一般"],
                  ["中年", "否", "否", "好"],
                  ["中年", "是", "是", "好"],
                  ["中年", "否", "是", "非常好"],
                  ["中年", "否", "是", "非常好"],
                  ["老年", "否", "是", "非常好"],
                  ["老年", "否", "是", "好"],
                  ["老年", "是", "否", "好"],
                  ["老年", "是", "否", "非常好"],
                  ["老年", "否", "否", "一般"]]),
        np.array(["否", "否", "是", "是", "否",
                  "否", "否", "是", "是", "是",
                  "是", "是", "是", "是", "否"])]

decision_tree = DecisionTreeC45WithoutPruning(X, Y, labels=["年龄", "有工作", "有自己的房子", "信贷情况"])
print(decision_tree)

有自己的房子 = 是 -> 是
有自己的房子 = 否 :
  有工作 = 是 -> 是
  有工作 = 否 -> 否
